In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

DATA = {
    "10 industries": "csv/10_industry.csv",
    "30 industries": "csv/30_industry.csv",
}

TRAIN_START = "2000-01-01"
TRAIN_END   = "2012-12-31"      # fold 1 training span
LB_LIST     = [252, 504]        # one year, two years
STEP        = 21                # roll one month at a time (overlapping)
HORIZON_CUR = 126               # the horizon in use, drawn as a reference line


def load_returns(path):
    df = pd.read_csv(path).set_index("Date")
    idx = pd.to_datetime(df.index, format="%Y%m%d", errors="coerce")
    if idx.isna().any():
        idx = pd.to_datetime(df.index, errors="coerce")
    df.index = idx
    df = df.sort_index()
    df = df[~df.index.duplicated(keep="first")]
    return df / 100.0


rets = {}
for name, path in DATA.items():
    d = load_returns(path).loc[TRAIN_START:TRAIN_END]
    rets[name] = d
    print(f"{name:15s}  {d.shape[0]:5d} days x {d.shape[1]:2d}  "
          f"{d.index[0].date()} ~ {d.index[-1].date()}")

In [ ]:
def mdd_episode(path):
    """
    Find the largest drawdown episode on a cumulative path and return its size, its
    decline length, and its length including recovery.

    path : 1D array, the cumulative return path

    Returns
    -------
    dict
      mdd          : maximum drawdown (absolute)
      peak, trough : indices
      dur_decline  : trading days from peak to trough
      dur_recovery : trading days from peak to recovery, NaN if it never recovers
    """
    path   = np.asarray(path, dtype=float)
    runmax = np.maximum.accumulate(path)
    dd     = runmax - path

    trough = int(np.argmax(dd))
    peak   = int(np.argmax(path[:trough + 1])) if trough > 0 else 0

    # recovery: the first point after the trough that regains the peak level
    after = np.where(path[trough:] >= path[peak])[0]
    dur_recovery = float(trough + after[0] - peak) if len(after) else np.nan

    return {
        "mdd"         : float(dd[trough]),
        "peak"        : peak,
        "trough"      : trough,
        "dur_decline" : float(trough - peak),
        "dur_recovery": dur_recovery,
    }


def cum_path(r, compounded=False):
    """Daily returns to a cumulative path. Additive by default (uncompounded), which
    matches the constraint in the paper."""
    return np.cumprod(1 + r) - 1 if compounded else np.cumsum(r)

In [ ]:
records = []

for ds_name, d in rets.items():
    R      = d.values                      # (T, m)
    dates  = d.index
    T, m   = R.shape
    ew     = R.mean(axis=1)                # equal-weighted portfolio

    for LB in LB_LIST:
        if T < LB:
            continue
        for s in range(0, T - LB + 1, STEP):
            e = s + LB
            win_start, win_end = dates[s], dates[e - 1]

            for compounded in (False, True):
                # -- equal-weighted portfolio --
                ep = mdd_episode(cum_path(ew[s:e], compounded))
                records.append({
                    "dataset": ds_name, "LB": LB,
                    "start": win_start, "end": win_end,
                    "series": "EW portfolio", "asset": "EW",
                    "compounded": compounded, **ep,
                })

                # -- per industry --
                for j, col in enumerate(d.columns):
                    ep = mdd_episode(cum_path(R[s:e, j], compounded))
                    records.append({
                        "dataset": ds_name, "LB": LB,
                        "start": win_start, "end": win_end,
                        "series": "Individual", "asset": col,
                        "compounded": compounded, **ep,
                    })

dur = pd.DataFrame(records)
print(f"{len(dur):,} episodes in total")
print(dur.groupby(["dataset", "LB", "series", "compounded"]).size()
         .rename("n").to_frame())

In [ ]:
def summarize(sub, col="dur_decline"):
    v = sub[col].dropna()
    return pd.Series({
        "n"      : len(v),
        "mean"   : v.mean(),
        "median" : v.median(),
        "p75"    : v.quantile(0.75),
        "p90"    : v.quantile(0.90),
        "p95"    : v.quantile(0.95),
        "max"    : v.max(),
        f"share <= {HORIZON_CUR} days": (v <= HORIZON_CUR).mean(),
    })

for col, tag in [("dur_decline", "decline (peak to trough)"),
                 ("dur_recovery", "including recovery (peak to recovery)")]:
    print(f"\n{'='*78}\n  {tag}  -- trading days\n{'='*78}")
    tbl = (dur[~dur["compounded"]]
           .groupby(["dataset", "LB", "series"])
           .apply(lambda g: summarize(g, col)))
    display(tbl.round(1))

In [ ]:
COL = "dur_decline"        # or "dur_recovery"

sub = dur[(~dur["compounded"]) & (dur["series"] == "EW portfolio")]

fig, axes = plt.subplots(len(DATA), len(LB_LIST),
                         figsize=(6.5 * len(LB_LIST), 4.2 * len(DATA)),
                         squeeze=False)

for i, ds_name in enumerate(DATA):
    for j, LB in enumerate(LB_LIST):
        ax = axes[i][j]
        v  = sub[(sub.dataset == ds_name) & (sub.LB == LB)][COL].dropna()
        if v.empty:
            ax.set_visible(False); continue

        ax.hist(v, bins=30, density=True, alpha=0.55,
                color="steelblue", edgecolor="white")
        sns.kdeplot(v, ax=ax, color="navy", lw=1.8)

        for q, c in [(0.50, "green"), (0.90, "orange")]:
            ax.axvline(v.quantile(q), color=c, ls="--", lw=1.4,
                       label=f"p{int(q*100)} = {v.quantile(q):.0f}d")
        ax.axvline(HORIZON_CUR, color="crimson", lw=2,
                   label=f"Current horizon = {HORIZON_CUR}d")

        yrs = "1Y" if LB == 252 else "2Y"
        ax.set_title(f"{ds_name} | Lookback = {LB}d ({yrs})", fontsize=11)
        ax.set_xlabel("MDD duration (trading days)")
        ax.set_ylabel("Density")
        ax.legend(fontsize=8)

fig.suptitle("Maximum Drawdown Duration Distribution  (2000-2012, EW portfolio)",
             fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()